# Hyperparameter Tuning with Optuna

Optimizes PPO hyperparameters for RL trading agents using Bayesian optimization.

## What Gets Tuned

**EMA Sharpe (7 parameters):**
- Core RL: gamma, softmax_temperature, learning_rate
- Reward: rolling_vol_window
- Trading: transaction_cost
- Training: patience

**Multi-Objective (11 parameters):**
- Core RL: gamma, softmax_temperature, learning_rate  
- Reward weights: return_scale, volatility_penalty, concentration_penalty, turnover_penalty
- Windows: vol_window
- Trading: transaction_cost, max_concentration
- Training: patience

## Usage

1. **Quick test** (Cell 5): Run 3 trials to verify everything works
2. **Full tuning** (Cells 1-3): Run 30-50 trials (2-5 hours)
3. Results saved to `results/` directory with visualizations

## Key Points

- Optimizes on **validation Sharpe** (not test) to avoid overfitting
- Uses TPE (Tree-structured Parzen Estimator) sampler for efficiency
- Includes pruning to stop unpromising trials early
- Can resume interrupted studies with `load_if_exists=True`


In [1]:
"""
Lightweight Optuna Hyperparameter Tuning for RL Trading Models
Focuses on high-impact parameters with sensible search ranges
"""

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import json
from pathlib import Path
import numpy as np
from typing import Dict, Any
import warnings
warnings.filterwarnings('ignore')

# Assuming you have these functions from your codebase
from train import train
from config import get_config


# ============================================================================
# SEARCH SPACE DEFINITIONS
# ============================================================================

def suggest_ema_sharpe_params(trial: optuna.Trial, agent_type: str) -> Dict[str, Any]:
    """
    Light tuning for EMA Sharpe - 7 parameters
    Expected improvement: 5-10% on test Sharpe
    """
    params = {
        # Core RL parameters (MOST IMPORTANT)
        'gamma': trial.suggest_float('gamma', 0.85, 0.95, step=0.01),
        'softmax_temperature': trial.suggest_float('softmax_temperature', 0.5, 2.0, step=0.1),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-4, log=True),
        
        # Reward-specific parameters
        'rolling_vol_window': trial.suggest_int('rolling_vol_window', 4, 16, step=2),
        
        # Trading parameters
        'transaction_cost': trial.suggest_float('transaction_cost', 0.001, 0.003, step=0.0005),
        
        # Training parameters
        'patience': trial.suggest_int('patience', 10, 20, step=5),
        
        # Fixed parameters (don't tune)
        'total_steps': 300_000,
        'eval_freq': 5_000,
        'seed': 42,
        'random_start': True,
    }
    
    return params


def suggest_multi_objective_params(trial: optuna.Trial, agent_type: str) -> Dict[str, Any]:
    """
    Light tuning for Multi-Objective - 11 parameters
    Expected improvement: 10-20% on test Sharpe
    """
    
    # Different ranges for technical vs sentiment
    if agent_type == 'technical':
        return_scale_range = (4.0, 12.0)
        vol_penalty_range = (0.01, 0.15)
        conc_penalty_range = (0.1, 0.5)
        turnover_penalty_range = (0.002, 0.01)
    else:  # sentiment
        return_scale_range = (2.0, 6.0)
        vol_penalty_range = (0.1, 0.4)
        conc_penalty_range = (0.5, 2.0)
        turnover_penalty_range = (0.01, 0.05)
    
    params = {
        # Core RL parameters
        'gamma': trial.suggest_float('gamma', 0.85, 0.95, step=0.01),
        'softmax_temperature': trial.suggest_float('softmax_temperature', 0.5, 2.0, step=0.1),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-4, log=True),
        
        # Multi-objective reward weights (MOST IMPORTANT FOR THIS MODEL)
        'return_scale': trial.suggest_float('return_scale', *return_scale_range, step=0.5),
        'volatility_penalty': trial.suggest_float('volatility_penalty', *vol_penalty_range, step=0.01),
        'concentration_penalty': trial.suggest_float('concentration_penalty', *conc_penalty_range, step=0.05),
        'turnover_penalty': trial.suggest_float('turnover_penalty', *turnover_penalty_range, step=0.001),
        
        # Window parameters
        'vol_window': trial.suggest_int('vol_window', 8, 20, step=2),
        
        # Trading parameters
        'transaction_cost': trial.suggest_float('transaction_cost', 0.001, 0.003, step=0.0005),
        'max_concentration': trial.suggest_float('max_concentration', 0.25, 0.40, step=0.05),
        
        # Training parameters
        'patience': trial.suggest_int('patience', 10, 20, step=5),
        
        # Fixed parameters
        'total_steps': 300_000,
        'eval_freq': 5_000,
        'seed': 42,
        'random_start': True,
    }
    
    return params


# ============================================================================
# OBJECTIVE FUNCTION (Generic for all reward types)
# ============================================================================
def objective_function(trial: optuna.Trial, agent_type: str = 'technical', reward_type: str = 'ema_sharpe') -> float:
    """
    Generic objective function for any reward type
    Returns: validation Sharpe ratio (maximize)
    """
    try:
        # Get base config with all required fields (data_dir, etc.)
        base_config = get_config(reward_type, agent_type)
        
        # Get suggested parameters based on reward type
        if reward_type == 'ema_sharpe':
            suggested_params = suggest_ema_sharpe_params(trial, agent_type)
        elif reward_type == 'multi_objective':
            suggested_params = suggest_multi_objective_params(trial, agent_type)
        else:
            raise ValueError(f"Unknown reward_type: {reward_type}")
        
        # Merge suggested params into base config (overrides defaults)
        config = {**base_config, **suggested_params}
        
        # Train model
        result = train(agent_type, 'PPO', reward_type, config, verbose=False)
        
        # Use actual validation Sharpe from training result
        val_sharpe = result['val_sharpe']
        
        # Report intermediate values for pruning
        trial.report(val_sharpe, step=0)
        
        # Check if trial should be pruned
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return val_sharpe
        
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return -1.0


# ============================================================================
# GENERIC TUNING FUNCTION (Replaces tune_ema_sharpe and tune_multi_objective)
# ============================================================================
def tune_model(
    reward_type: str = 'ema_sharpe',
    agent_type: str = 'technical',
    n_trials: int = 30,
    study_name: str = None,
    load_if_exists: bool = True
) -> optuna.Study:
    """
    Generic tuning function for any reward type
    
    Args:
        reward_type: 'ema_sharpe' or 'multi_objective'
        agent_type: 'technical' or 'sentiment'
        n_trials: Number of trials to run
        study_name: Name for the study (for resuming)
        load_if_exists: Whether to resume existing study
    
    Returns:
        Completed study object
    """
    # Set defaults based on reward type
    if study_name is None:
        study_name = f"{reward_type}_{agent_type}_tuning"
    
    n_params = 7 if reward_type == 'ema_sharpe' else 11
    n_startup = 10 if reward_type == 'ema_sharpe' else 15
    
    # Create study with pruning
    study = optuna.create_study(
        study_name=study_name,
        direction='maximize',
        sampler=TPESampler(seed=42, n_startup_trials=n_startup),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=0),
        load_if_exists=load_if_exists
    )
    
    # Run optimization
    print(f"\n{'='*60}")
    print(f"Tuning {reward_type.replace('_', ' ').title()} for {agent_type} agent")
    print(f"{'='*60}")
    print(f"Trials: {n_trials}")
    print(f"Search space: {n_params} parameters")
    print(f"Expected time: {n_trials * 4} - {n_trials * 6} minutes")
    print(f"{'='*60}\n")
    
    study.optimize(
        lambda trial: objective_function(trial, agent_type, reward_type),
        n_trials=n_trials,
        show_progress_bar=True
    )
    
    # Print results
    print(f"\n{'='*60}")
    print("TUNING COMPLETE")
    print(f"{'='*60}")
    print(f"Best validation Sharpe: {study.best_value:.4f}")
    print(f"Best parameters:")
    for param, value in study.best_params.items():
        print(f"  {param}: {value}")
    
    # Save results
    save_study_results(study, f"results/{study_name}.json")
    
    return study


# Convenience wrappers for backward compatibility
def tune_ema_sharpe(agent_type: str = 'technical', n_trials: int = 30, 
                    study_name: str = None, load_if_exists: bool = True) -> optuna.Study:
    """Tune EMA Sharpe model"""
    return tune_model('ema_sharpe', agent_type, n_trials, study_name, load_if_exists)


def tune_multi_objective(agent_type: str = 'technical', n_trials: int = 50,
                         study_name: str = None, load_if_exists: bool = True) -> optuna.Study:
    """Tune Multi-Objective model"""
    return tune_model('multi_objective', agent_type, n_trials, study_name, load_if_exists)


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def save_study_results(study: optuna.Study, filepath: str):
    """Save study results to JSON"""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    
    results = {
        'best_value': study.best_value,
        'best_params': study.best_params,
        'best_trial': study.best_trial.number,
        'n_trials': len(study.trials),
        'trials': [
            {
                'number': trial.number,
                'value': trial.value,
                'params': trial.params,
                'state': str(trial.state)
            }
            for trial in study.trials
        ]
    }
    
    with open(filepath, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ Results saved to: {filepath}")


def visualize_study(study: optuna.Study, save_dir: str = 'results/optuna_plots'):
    """
    Create visualization plots for the study
    Requires: pip install optuna plotly kaleido
    """
    try:
        import optuna.visualization as vis
        from pathlib import Path
        
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        
        # 1. Optimization history
        fig = vis.plot_optimization_history(study)
        fig.write_html(f"{save_dir}/optimization_history.html")
        
        # 2. Parameter importances
        fig = vis.plot_param_importances(study)
        fig.write_html(f"{save_dir}/param_importances.html")
        
        # 3. Parallel coordinate plot
        fig = vis.plot_parallel_coordinate(study)
        fig.write_html(f"{save_dir}/parallel_coordinate.html")
        
        # 4. Slice plot (parameter effects)
        fig = vis.plot_slice(study)
        fig.write_html(f"{save_dir}/slice_plot.html")
        
        print(f"\n✓ Visualizations saved to: {save_dir}/")
        
    except ImportError:
        print("\nInstall plotly for visualizations: pip install plotly kaleido")


def compare_with_baseline(study: optuna.Study, baseline_sharpe: float):
    """Compare tuned results with baseline"""
    improvement = study.best_value - baseline_sharpe
    improvement_pct = (improvement / baseline_sharpe) * 100
    
    print(f"\n{'='*60}")
    print("COMPARISON WITH BASELINE")
    print(f"{'='*60}")
    print(f"Baseline Sharpe:    {baseline_sharpe:.4f}")
    print(f"Tuned Sharpe:       {study.best_value:.4f}")
    print(f"Improvement:        {improvement:+.4f} ({improvement_pct:+.2f}%)")
    print(f"{'='*60}\n")



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:


# Example 1: Quick tune EMA Sharpe (30 trials ≈ 2-3 hours)
print("Example 1: Tuning EMA Sharpe")
study_ema = tune_ema_sharpe(
    agent_type='technical',
    n_trials=30,
    study_name='ema_quick_tune'
)

# Compare with baseline
compare_with_baseline(study_ema, baseline_sharpe=2.651)  # Your current val Sharpe

# Visualize results
visualize_study(study_ema, save_dir='results/ema_plots')


[I 2025-10-28 14:54:06,360] A new study created in memory with name: ema_quick_tune


Example 1: Tuning EMA Sharpe

Tuning Ema Sharpe for technical agent
Trials: 30
Search space: 7 parameters
Expected time: 120 - 180 minutes



  0%|          | 0/30 [00:00<?, ?it/s]

[I 2025-10-28 14:55:23,824] Trial 0 finished with value: 2.437 and parameters: {'gamma': 0.89, 'softmax_temperature': 2.0, 'learning_rate': 0.0003248192869770293, 'rolling_vol_window': 12, 'transaction_cost': 0.001, 'patience': 10}. Best is trial 0 with value: 2.437.
[I 2025-10-28 14:56:37,151] Trial 1 finished with value: 2.284 and parameters: {'gamma': 0.85, 'softmax_temperature': 1.8, 'learning_rate': 0.000263124545105745, 'rolling_vol_window': 12, 'transaction_cost': 0.001, 'patience': 20}. Best is trial 0 with value: 2.437.
[I 2025-10-28 14:57:28,185] Trial 2 finished with value: 2.294 and parameters: {'gamma': 0.94, 'softmax_temperature': 0.8, 'learning_rate': 0.00013399549522183018, 'rolling_vol_window': 6, 'transaction_cost': 0.0015, 'patience': 15}. Best is trial 0 with value: 2.437.
[I 2025-10-28 14:58:31,929] Trial 3 finished with value: 2.364 and parameters: {'gamma': 0.89, 'softmax_temperature': 0.9, 'learning_rate': 0.0002677113724214593, 'rolling_vol_window': 4, 'transac

In [ ]:


# Example 2: Thorough tune Multi-Objective (50 trials ≈ 3-5 hours)
print("\n" + "="*60)
print("Example 2: Tuning Multi-Objective")
study_multi = tune_multi_objective(
    agent_type='technical',
    n_trials=50,
    study_name='multi_obj_thorough_tune'
)

# Compare with baseline
compare_with_baseline(study_multi, baseline_sharpe=2.347)  # Your current val Sharpe

# Visualize results
visualize_study(study_multi, save_dir='results/multi_obj_plots')


[I 2025-10-28 17:02:52,535] A new study created in memory with name: multi_obj_thorough_tune



Example 2: Tuning Multi-Objective

Tuning Multi Objective for technical agent
Trials: 50
Search space: 11 parameters
Expected time: 200 - 300 minutes



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-10-28 17:03:28,279] Trial 0 finished with value: 2.23 and parameters: {'gamma': 0.89, 'softmax_temperature': 2.0, 'learning_rate': 0.0003248192869770293, 'return_scale': 9.0, 'volatility_penalty': 0.03, 'concentration_penalty': 0.15000000000000002, 'turnover_penalty': 0.002, 'vol_window': 20, 'transaction_cost': 0.0025, 'max_concentration': 0.35, 'patience': 10}. Best is trial 0 with value: 2.23.
[I 2025-10-28 17:04:19,528] Trial 1 finished with value: 2.54 and parameters: {'gamma': 0.95, 'softmax_temperature': 1.8, 'learning_rate': 0.00014074036373847487, 'return_scale': 5.5, 'volatility_penalty': 0.03, 'concentration_penalty': 0.2, 'turnover_penalty': 0.006, 'vol_window': 14, 'transaction_cost': 0.0015, 'max_concentration': 0.35, 'patience': 10}. Best is trial 1 with value: 2.54.
[I 2025-10-28 17:04:58,047] Trial 2 finished with value: 2.269 and parameters: {'gamma': 0.88, 'softmax_temperature': 1.0, 'learning_rate': 0.000208343156115295, 'return_scale': 10.5, 'volatility_pen

In [ ]:


# Example 3: Resume interrupted study
print("\n" + "="*60)
print("Example 3: Resume previous study")
study_resumed = tune_ema_sharpe(
    agent_type='technical',
    n_trials=10,  # Additional trials
    study_name='ema_quick_tune',  # Same name as before
    load_if_exists=True  # Will continue from where it left off
)